# 06 - Interpretasi Hasil dan Rekomendasi

**Tahap 6** metodologi: menggabungkan seluruh hasil menjadi kesimpulan dan
rekomendasi yang dapat dipakai untuk pengambilan keputusan.

Notebook ini menjawab dua pertanyaan:

1. **Apa yang sudah baik** di Silol Kopi & Eatery?
2. **Apa yang perlu diperbaiki**, dan mana yang paling mendesak?

In [1]:
import sys
from pathlib import Path

# Menemukan root proyek agar `import src` berjalan dari mana pun notebook dibuka.
ROOT = Path.cwd()
if not (ROOT / "src").exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

import warnings
warnings.filterwarnings("ignore")

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from src import config

pd.set_option("display.max_columns", 40)
pd.set_option("display.width", 160)
pd.set_option("display.max_colwidth", 90)

print("Root proyek:", ROOT)

Root proyek: D:\hikam\tugas DataMining


In [2]:
from src import interpret, io_utils

clustered = io_utils.load_csv(config.CLUSTERED_CSV)
aspect_summary = io_utils.load_csv(config.ASPECT_SUMMARY_CSV)
cluster_profile = io_utils.load_csv(config.CLUSTER_SUMMARY_CSV)
hits_path = config.ASPECT_SENTIMENT_CSV
hits = io_utils.load_csv(hits_path) if hits_path.exists() else None

print(f"Ulasan: {len(clustered):,} | Aspek: {len(aspect_summary)} | Cluster: {clustered['cluster'].nunique()}")

Ulasan: 6,795 | Aspek: 9 | Cluster: 4


## 6.1 Ringkasan eksekutif

In [3]:
summary = interpret.executive_summary(clustered, aspect_summary)
io_utils.save_csv(summary, config.TABLES_DIR / "06_ringkasan_eksekutif.csv")
summary

,Indikator,Nilai
0,Jumlah ulasan dianalisis,6795
1,Rating rata-rata,4.252
2,Sentimen Positif,4610 (67.84%)
3,Sentimen Netral,1275 (18.76%)
4,Sentimen Negatif,910 (13.39%)
5,Aspek paling perlu diperbaiki,Kecepatan (57.1% negatif)
6,Aspek terbaik,Kebersihan (89.0% positif)
7,Jumlah aspek terdeteksi,9


## 6.2 Aspek yang sudah baik (kekuatan)

Aspek dengan persentase positif tinggi dan volume penyebutan besar.

In [4]:
strengths = aspect_summary.sort_values(["% Positif", "Jumlah Disebut"], ascending=False)
strengths[["Aspek", "Jumlah Disebut", "% Positif", "% Negatif", "Rating Rata-rata"]].head()

,Aspek,Jumlah Disebut,% Positif,% Negatif,Rating Rata-rata
8,Kebersihan,137,89.05,6.57,4.20
7,Harga,1207,83.76,10.44,4.31
6,Suasana,4240,81.82,7.22,4.48
4,Pelayanan,1010,69.11,17.03,3.30
1,Makanan,2244,68.00,17.20,3.81


## 6.3 Aspek yang perlu diperbaiki

Diurutkan berdasarkan skor prioritas (kombinasi volume keluhan dan persentase
negatif).

In [5]:
priority = interpret.priority_matrix(aspect_summary)
io_utils.save_csv(priority, config.TABLES_DIR / "06_matriks_prioritas.csv")
priority

,Aspek,Jumlah Disebut,% Negatif,% Positif,Rating Rata-rata,Hidden Complaint,Skor Prioritas,Peringkat Prioritas,Rekomendasi,Kuadran
0,Kecepatan,690,57.10,31.16,3.02,116,14.999,1,Prioritas 1 - Perbaikan mendesak,Temuan Khusus (jarang dibahas tapi negatif)
1,Makanan,2244,17.20,68.00,3.81,189,8.148,2,Prioritas 1 - Perbaikan mendesak,Prioritas Utama (sering dibahas & banyak negatif)
2,Minuman,1299,17.24,53.19,4.07,117,6.214,3,Prioritas 1 - Perbaikan mendesak,Prioritas Utama (sering dibahas & banyak negatif)
3,Fasilitas,627,24.56,35.25,4.11,105,6.150,4,Prioritas 2 - Perlu ditingkatkan,Temuan Khusus (jarang dibahas tapi negatif)
4,Pelayanan,1010,17.03,69.11,3.30,33,5.412,5,Prioritas 2 - Perlu ditingkatkan,Kekuatan (sering dipuji)
5,Parkir,712,19.94,59.55,4.06,81,5.321,6,Prioritas 2 - Perlu ditingkatkan,Temuan Khusus (jarang dibahas tapi negatif)
6,Suasana,4240,7.22,81.82,4.48,234,4.701,7,Pertahankan,Kekuatan (sering dipuji)
7,Harga,1207,10.44,83.76,4.31,76,3.627,8,Prioritas 3 - Pantau,Kekuatan (sering dipuji)
8,Kebersihan,137,6.57,89.05,4.20,4,0.769,9,Pertahankan,"Minor (jarang dibahas, sedikit negatif)"


In [6]:
from src import visualize

visualize.plot_priority_bar(priority)

WindowsPath('D:/hikam/tugas DataMining/results/figures/06_prioritas.png')

## 6.4 Tema ulasan hasil clustering

K-Means mengelompokkan ulasan berdasarkan kemiripan teks. Tema diberi nama
otomatis dari term dominan tiap cluster dan dipetakan ke aspek layanan.

In [7]:
cluster_profile[[
    "Cluster", "Jumlah Ulasan", "Persentase", "Rating Rata-rata",
    "Tema (otomatis)", "Sentimen Dominan", "Term Dominan",
]]

,Cluster,Jumlah Ulasan,Persentase,Rating Rata-rata,Tema (otomatis),Sentimen Dominan,Term Dominan
0,2,4882,71.85,4.126,Suasana (beragam),Positif,"nyaman, layan, makan, bagus, harga, kopi, nongkrong, ramai"
1,0,912,13.42,4.634,Live Music & Suasana,Positif,"live musik, musik, live, nongkrong, nongkrong live, makan, enak, asik"
2,1,646,9.51,4.432,Makanan (beragam),Positif,"enak, makan, makan enak, nyaman, kopi enak, kopi, harga, minum"
3,3,355,5.22,4.679,Suasana,Positif,"nongkrong, enak nongkrong, asik nongkrong, asik, enak, cocok nongkrong, cocok, nyaman ..."


## 6.5 Hubungan cluster dengan aspek layanan

Bila cluster hasil K-Means dan aspek hasil ekstraksi otomatis saling mendukung,
kesimpulan menjadi lebih kuat.

In [8]:
if hits is not None and not hits.empty:
    matrix_ca = interpret.cluster_aspect_matrix(
        clustered, clustered["cluster"].to_numpy(), hits
    )
    io_utils.save_csv(matrix_ca, config.TABLES_DIR / "06_cluster_x_aspek.csv")
    visualize.plot_cluster_aspect_heatmap(matrix_ca)
    matrix_ca
else:
    print("Data sentimen aspek belum tersedia.")

## 6.6 Rekomendasi tindakan per aspek

In [9]:
recs = interpret.recommendations(aspect_summary)
io_utils.save_csv(recs, config.TABLES_DIR / "06_rekomendasi.csv")
recs

,Aspek,Peringkat,Prioritas,% Negatif,Jumlah Disebut,Rekomendasi Tindakan
0,Kecepatan,1,Prioritas 1 - Perbaikan mendesak,57.10,690,Evaluasi alur dapur dan koordinasi antar pelayan; terapkan target waktu saji per menu.
1,Makanan,2,Prioritas 1 - Perbaikan mendesak,17.20,2244,Kontrol konsistensi rasa dan porsi; uji rasa berkala pada menu paling sering dipesan.
2,Minuman,3,Prioritas 1 - Perbaikan mendesak,17.24,1299,Standarkan resep dan suhu penyajian; lakukan quality check rasa minuman.
3,Fasilitas,4,Prioritas 2 - Perlu ditingkatkan,24.56,627,"Periksa kelengkapan dan kondisi fasilitas (ruang, meja, colokan, area VIP) secara berk..."
4,Pelayanan,5,Prioritas 2 - Perlu ditingkatkan,17.03,1010,Pelatihan keramahan dan responsivitas staf; perjelas SOP penanganan keluhan.
5,Parkir,6,Prioritas 2 - Perlu ditingkatkan,19.94,712,Tertibkan tarif dan juru parkir; sediakan area parkir tambahan pada jam ramai.
6,Suasana,7,Pertahankan,7.22,4240,Pertahankan konsep dan live music; jaga kenyamanan akustik saat ramai.
7,Harga,8,Prioritas 3 - Pantau,10.44,1207,"Komunikasikan value (porsi, fasilitas) dan tinjau harga menu yang dianggap mahal."
8,Kebersihan,9,Pertahankan,6.57,137,Tambahkan jadwal pembersihan toilet dan area makan; audit kebersihan mingguan.


## 6.7 Bukti pendukung: kutipan pelanggan per aspek prioritas

In [10]:
examples = io_utils.load_csv(config.TABLES_DIR / "04_contoh_aspek.csv")
neg = examples[examples["Jenis"] == "Negatif"]

for aspect in priority["Aspek"].head(4):
    rows = neg[neg["Aspek"] == aspect].head(3)
    if rows.empty:
        continue
    print(f"\n### {aspect}")
    for _, r in rows.iterrows():
        print(f"  • [{r['Rating']}★] {r['Klausa'][:180]}")


### Kecepatan
  • [1★] menunggu satu pesanan lama sekali dari kafe sepi sampai ramai tamunya
  • [1★] sistem pesan dan bayarnya sangat ribet
  • [4★] pesan strawberry tea berasa pesan teh tawar  tidak ada manis sama sekali

### Makanan
  • [4★] kasian meja sebelah sudah menunggu lama dan menu yang dipesan soldout telat dikabari
  • [3★] minimal goreng nya yang kering biar bisa crispy misal tidak dibuat krispi minimal jangan hancur tahunya   beet thai salad beef nya keras kalii lah
  • [3★] makannya tidak enak pelayanan agak lama hingar bingar banget karena untuk "konser" musik yang deket banget sama meja karena memang indoor

### Minuman
  • [5★] non alkohol untuk yang mencari ketenangan saat minum kopi kafe ini tak rekomendasi
  • [1★] hanya jualan tempat pelayanan buruk rasa makanan buruk waktu menunggu makanan dan minuman sangat lama
  • [2★] pelayanan yang agak cukup lama di sini saya memesan minuman harus menunggu lama hingga semua selesai makan habis dan di tanya lagi kita mesen

## 6.8 Kesimpulan akhir

Isi bagian ini secara naratif di laporan, dengan angka dari tabel di atas.
Kerangka yang disarankan:

1. **Karakteristik data** — jumlah ulasan yang dianalisis, sebaran rating,
   dan seberapa banyak ulasan tanpa teks.
2. **Sentimen keseluruhan** — proporsi positif/netral/negatif, serta tingkat
   kesesuaian antara teks dan rating.
3. **Tema ulasan** — hasil clustering beserta term dominan tiap cluster.
4. **Aspek layanan** — aspek yang menjadi kekuatan dan aspek yang menjadi
   keluhan, lengkap dengan persentase.
5. **Prioritas perbaikan** — urutan aspek berdasarkan skor prioritas, disertai
   kutipan pelanggan.
6. **Rekomendasi** — tindakan konkret untuk setiap aspek prioritas.

### Keterbatasan

- Label sentimen berbasis lexicon tidak memahami sarkasme dan konteks yang
  sangat spesifik.
- Sebagian besar ulasan sangat pendek sehingga struktur cluster lemah
  (Silhouette rendah) — hal ini wajar pada data ulasan.
- Penentuan ambang frekuensi aspek memengaruhi jumlah aspek yang terbentuk;
  ambang perlu dilaporkan bersama hasilnya agar dapat direproduksi.